<div style="padding: 1em 0.5em; color: #fff; background-color: #0969da; font-size: 1.2em;">
    Jour 3c — Étudiant hybride k-mer + CNN avec distillation
</div>
<div style="border-left: 2px solid #0969da; min-height: 1.5em;margin-left: 1em;padding: 1em;">
    - Combiner les caractéristiques k-mer artisanales avec les motifs CNN appris via fusion tardive<br>
    - Améliorer le modèle hybride avec la distillation de connaissances à partir du professeur Evo2<br>
    - Analyser les poids de fusion appris et comparer avec les modèles de référence<br>
</div>

#### **Votre identité**

Double-cliquez sur cette cellule et complétez, puis exécutez-la (`Maj + Entrée`).

- **Nom & prénom :** _à compléter_
- **Groupe / binôme :** _à compléter_
- **Date :** _à compléter_

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **Introduction**

Le student du notebook précédent (k-mer + MLP) a atteint ses limites de représentation.
Ici, nous combinons deux représentations complémentaires :

- **k-mers** : caractéristiques artisanales, robustes mais perdent l'ordre des nucléotides
- **CNN one-hot** : apprend des motifs locaux et positionnels directement depuis la séquence

La **fusion tardive** combine ces deux flux après leur encodage, avec des poids appris.
La distillation guide cette fusion avec les cibles douces du teacher Evo2.

**Configuration optimale (basée sur les tests)** :
- Temperature = 2.0
- Alpha = 0.3
- Dropout = 0.3

**Attendu :** ~0.84 de précision (meilleur que le student k-mer seul à ~0.84)

<img src="assets/Hybrid k-mer CNN Distillation Pipeline.png" width="600"/>

In [ ]:
import sys, torch, torch.nn as F, torch.nn as nn
from pathlib import Path
sys.path.append("src")
from data import load_all
from embeddings import load_supervised_embeddings
from featurize import kmer_matrix, one_hot_batch
from models.classifier_heads import MLPHead
from models.baselines import OneHotCNN
from eval import evaluate_logits, count_params
EMB_DIR, PROCESSED_DIR = "../2-data/embeddings", "../2-data/processed"
TEACHER_PATH = Path("../2-data/models/teacher_mlp.pt")

In [ ]:
X_train_emb, y_train, ids_train = load_supervised_embeddings(EMB_DIR, "train")
X_val_emb, y_val, ids_val = load_supervised_embeddings(EMB_DIR, "val")
ckpt = torch.load(TEACHER_PATH)
teacher = MLPHead(d_in=ckpt["d_in"]).load_state_dict(ckpt["state_dict"]).eval()
print("teacher rechargé depuis", TEACHER_PATH)

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **Préparer les données : aligner les séquences et calculer les caractéristiques**

In [ ]:
# TODO 1 : récupérez les séquences correspondant aux ids_train et ids_val, calculez les matrices de k-mer (k=4) et encodez en one-hot (WINDOW=200)
splits = load_all(PROCESSED_DIR, max_rows=None)
train_seqs = splits["train"].set_index("id").loc[ids_train, "sequence"].tolist()
val_seqs = splits["val"].set_index("id").loc[ids_val, "sequence"].tolist()
X_train_kmer = torch.tensor(kmer_matrix(train_seqs, k=4), dtype=torch.float32)
X_val_kmer = torch.tensor(kmer_matrix(val_seqs, k=4), dtype=torch.float32)
WINDOW = 200
X_train_oh = torch.tensor(one_hot_batch(train_seqs, length=WINDOW))
X_val_oh = ...
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)
print("k-mer shape:", X_train_kmer.shape, "one-hot shape:", X_train_oh.shape)

In [ ]:
# TODO 2 : calculez les logits du teacher sur les données d'entraînement et validation
with torch.no_grad():
    teacher_logits_train = teacher(torch.tensor(X_train_emb, dtype=torch.float32))
    teacher_val_logits = ...
teacher_metrics = evaluate_logits(teacher_val_logits, y_val)
print("teacher metrics:", teacher_metrics)

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **Définir le modèle hybride avec fusion tardive**

In [ ]:
# TODO 3 : définissez le modèle hybride avec fusion tardive (k-mer MLP + CNN + fusion pondérée + classifieur)
class HybridKmerCNN(nn.Module):
    def __init__(self, kmer_dim, seq_len, fusion_dim=128, dropout=0.3, cnn_channels=32):
        super().__init__()
        self.kmer_mlp = nn.Sequential(nn.Linear(kmer_dim, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, fusion_dim))
        self.cnn = OneHotCNN(seq_len=seq_len, channels=cnn_channels)
        self.cnn_proj = nn.Sequential(nn.Linear(cnn_channels, fusion_dim), nn.ReLU(), nn.Dropout(dropout))
        self.fusion_weight = nn.Parameter(torch.tensor([0.5, 0.5]))
        self.classifier = nn.Sequential(nn.Linear(fusion_dim, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x_kmer, x_oh):
        kmer_feat = self.kmer_mlp(x_kmer)
        cnn_feat = self.cnn_proj(self.cnn.net(x_oh).squeeze(-1))
        w = torch.softmax(self.fusion_weight, dim=0)
        fused = ...
        return self.classifier(fused)

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **Entraîner le student hybride avec distillation**

In [ ]:
# Configuration d'entraînement (basée sur les meilleurs résultats des tests)
N_EPOCHS = 100
TEMPERATURE = 2.0
ALPHA = 0.3
LR = 1e-3
DROPOUT = 0.3

In [ ]:
# TODO 4 : instanciez le modèle hybride et l'optimiseur
hybrid = ...
optimizer = torch.optim.Adam(hybrid.parameters(), lr=LR)
print(f"Hybrid model parameters: {count_params(hybrid):,}")

In [ ]:
# TODO 5 : implémentez la boucle d'entraînement avec distillation (loss_hard + loss_soft avec TEMPERATURE et ALPHA)
n, batch_size = X_train_kmer.shape[0], 256
for epoch in range(N_EPOCHS):
    for start in range(0, n, batch_size):
        idx = torch.randperm(n)[start:start+batch_size]
        optimizer.zero_grad()
        student_logits = hybrid(X_train_kmer[idx], X_train_oh[idx])
        loss_hard = F.binary_cross_entropy_with_logits(student_logits.squeeze(), y_train_t[idx])
        teacher_logits_2d = torch.stack([-teacher_logits_train[idx], teacher_logits_train[idx]], dim=1)
        student_logits_2d = torch.stack([-student_logits.squeeze(), student_logits.squeeze()], dim=1)
        loss_soft = F.kl_div(F.log_softmax(student_logits_2d/TEMPERATURE, dim=1), F.softmax(teacher_logits_2d/TEMPERATURE, dim=1), reduction='batchmean') * (TEMPERATURE**2)
        loss = ...
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"epoch {epoch+1}/{N_EPOCHS}")

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **Évaluer le student hybride**

In [ ]:
hybrid.eval()
with torch.no_grad():
    val_logits = hybrid(X_val_kmer, X_val_oh).numpy()
hybrid_metrics = evaluate_logits(val_logits, y_val)
print("hybrid student:", hybrid_metrics, "params:", count_params(hybrid))
import time
hybrid.eval()
with torch.no_grad():
    start = time.time()
    for _ in range(100): _ = hybrid(X_val_kmer[:1], X_val_oh[:1])
    latency_ms = (time.time() - start) * 1000 / 100
print(f"latency (ms/sample): {latency_ms:.4f}")

In [ ]:
fusion_weights = torch.softmax(hybrid.fusion_weight, dim=0).detach().numpy()
print(f"Fusion weights: k-mer={fusion_weights[0]:.3f}, CNN={fusion_weights[1]:.3f}")

# Entraîner le même modèle hybride SANS distillation (alpha=1.0) pour comparer
hybrid_sup = HybridKmerCNN(kmer_dim=X_train_kmer.shape[1], seq_len=WINDOW, dropout=DROPOUT)
optimizer_sup = torch.optim.Adam(hybrid_sup.parameters(), lr=LR)
for epoch in range(N_EPOCHS):
    for start in range(0, n, batch_size):
        idx = torch.randperm(n)[start:start+batch_size]
        optimizer_sup.zero_grad()
        student_logits = hybrid_sup(X_train_kmer[idx], X_train_oh[idx])
        loss = F.binary_cross_entropy_with_logits(student_logits.squeeze(), y_train_t[idx])
        loss.backward()
        optimizer_sup.step()
hybrid_sup.eval()
with torch.no_grad():
    val_logits_sup = hybrid_sup(X_val_kmer, X_val_oh).numpy()
hybrid_sup_metrics = evaluate_logits(val_logits_sup, y_val)
print(f"hybrid sans distillation: accuracy={hybrid_sup_metrics['accuracy']:.3f}")

<div style="height: 3px; margin: 2em 0 1.5em 0; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>

#### **Comparaison avec les modèles de référence**

In [ ]:
import pandas as pd
comparison = pd.DataFrame([
    {"model": "teacher (Evo2+MLP)", **teacher_metrics, "params": count_params(teacher), "latency_ms": 0.15},
    {"model": "hybrid kmer+CNN (sans distillation)", **hybrid_sup_metrics, "params": count_params(hybrid_sup), "latency_ms": latency_ms},
    {"model": "hybrid kmer+CNN (avec distillation)", **hybrid_metrics, "params": count_params(hybrid), "latency_ms": latency_ms},
])
comparison

#### **Point de contrôle**

Le modèle hybride devrait :
- Avoir une précision d'environ 0.84 (meilleur que le student k-mer seul à ~0.84)
- Être beaucoup plus petit que le teacher (quelques milliers vs millions de paramètres)
- Être rapide à l'inférence (pas besoin d'Evo2)
- Montrer des poids de fusion équilibrés entre k-mers et CNN

La distillation aide principalement à guider la fusion des deux représentations,
permettant au modèle d'apprendre quand faire confiance à chaque branche.

Suite : `04_compression_analysis_and_wrapup.ipynb` — mettre tous les modèles sur un seul graphique.

*Bloqué ? La version complète est dans `solution/03_hybrid_kmer_cnn_distillation.ipynb`.*

<div style="margin-top: 3em;">
  <div style="height: 3px; background: linear-gradient(90deg, #0969da 0%, #0969da 55%, rgba(9,105,218,0.15) 100%);"></div>
  <div style="display: flex; align-items: center; gap: 0.9em; padding: 1.1em 1em; font-size: 0.9em; color: #57606a; background-color: #f2f6fd; border-radius: 0 0 6px 6px;">
    <svg width="34" height="34" viewBox="0 0 34 34" fill="none" style="flex: 0 0 auto;">
      <path d="M9 3c0 7 16 7 16 14S9 24 9 31" stroke="#0969da" stroke-width="2" stroke-linecap="round"/>
      <path d="M25 3c0 7-16 7-16 14s16 7 16 14" stroke="#0969da" stroke-width="2" stroke-linecap="round" opacity="0.45"/>
      <circle cx="17" cy="10" r="1.8" fill="#0969da"/>
      <circle cx="17" cy="24" r="1.8" fill="#0969da"/>
    </svg>
    <div style="flex: 1 1 auto;">
      <div style="color: #0969da; font-weight: 600; letter-spacing: 0.03em;">Fin du Jour 3c</div>
      <div>Prochaine &eacute;tape &rarr; <code>day3/04_compression_analysis_and_wrapup.ipynb</code></div>
    </div>
    <div style="flex: 0 0 auto; text-align: right; border-right: 2px solid #0969da; padding-right: 0.9em;">
      <div style="font-weight: 600; color: #24292f;">EEIA &middot; bioAI Workshop</div>
      <div style="font-size: 0.85em;">Semaine 4 &mdash; De l'ADN aux mod&egrave;les compress&eacute;s</div>
    </div>
  </div>
</div>